# BrailleLens — cell detector (Stage 4a)

Train single-class YOLO on Braille **cells**. GPU runtime required.

Source of truth: repo root `colab_training.md` **Job A**. Use a **separate** notebook for Stage 4b (CNN).

Upload `braille_cells.zip` to Drive:
`MyDrive/BrailleLens_Colab/4a_cell_detector/braille_cells.zip`

In [ ]:
!pip -q install ultralytics pyyaml opencv-python-headless pillow

In [ ]:
from google.colab import drive
from pathlib import Path
import torch, zipfile, yaml

assert torch.cuda.is_available(), "Switch Runtime to GPU and reconnect"
drive.mount("/content/drive")

src = Path("/content/drive/MyDrive/BrailleLens_Colab/4a_cell_detector/braille_cells.zip")
assert src.exists(), src
out = Path("/content/braille_cells")
if not (out / "data.yaml").exists():
    with zipfile.ZipFile(src) as z:
        z.extractall("/content")

yaml_path = out / "data.yaml"
cfg = yaml.safe_load(yaml_path.read_text(encoding="utf-8"))
cfg["path"] = str(out)
yaml_path.write_text(yaml.safe_dump(cfg, sort_keys=False), encoding="utf-8")

print("cuda", torch.cuda.get_device_name(0))
print("train/val/test pages",
      len(list((out / "images/train").glob("*"))),
      len(list((out / "images/val").glob("*"))),
      len(list((out / "images/test").glob("*"))))

In [ ]:
from ultralytics import YOLO
from pathlib import Path

run_dir = Path("/content/drive/MyDrive/BrailleLens_Colab/4a_cell_detector/runs")
run_dir.mkdir(parents=True, exist_ok=True)

model = YOLO("yolo26n.pt")
model.train(
    data="/content/braille_cells/data.yaml",
    epochs=80,
    imgsz=1280,
    batch=4,
    device=0,
    workers=2,
    patience=15,
    seed=42,
    max_det=800,
    fliplr=0.0,
    flipud=0.0,
    mosaic=0.5,
    mixup=0.0,
    project=str(run_dir),
    name="braille_cell_yolo26",
    exist_ok=True,
    save_period=5,
)

In [ ]:
import shutil
from pathlib import Path

src = Path("/content/drive/MyDrive/BrailleLens_Colab/4a_cell_detector/runs/braille_cell_yolo26/weights/best.pt")
dst = Path("/content/drive/MyDrive/BrailleLens_Colab/4a_cell_detector/braille_cell_best.pt")
assert src.exists(), src
shutil.copy2(src, dst)
print("send this file back:", dst, "bytes", dst.stat().st_size)